# Snacka neural v2: raw corpus preparation

CPU only. Downloads FLEURS Swedish (16 kHz) and VCTK (48 kHz), creates fixed train/validation/test manifests and persists the corpus on your Drive. This notebook does not train a model. FLEURS separation is by sentence group; VCTK separation is by speaker. Existing pilot artifacts in the same Drive folder can be used without rerunning acquisition.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import shutil, subprocess, tempfile, tarfile, json
assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "ffmpeg/ffprobe required"
DRIVE_ROOT = Path("/content/drive/MyDrive/Snacka/neural-v2")
assert DRIVE_ROOT.is_dir(), "Select the Drive folder containing this notebook"
# False: 180-clip pilot. True: all available source utterances, on cloud CPU.
FULL = False


The acquisition code below is embedded so the notebook does not depend on an unpushed branch or a mutable GitHub main. It never reads or uploads OAuth tokens. Full preparation can fetch most of VCTK and the 1.46 GB FLEURS train archive; keep this runtime on CPU.


In [ ]:
CORPUS_SCRIPT = '#!/usr/bin/env python3\n# SPDX-License-Identifier: MIT\n"""Build a native-rate, split-frozen raw corpus; no model training or credentials.\n\nRequires ffmpeg/ffprobe. Default pilot: 108 Swedish + 72 English utterances.\nUse --full on a cloud CPU to remove caps. Output is intentionally outside Git.\n"""\nimport argparse\nfrom collections import Counter, OrderedDict\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport re\nimport subprocess\nimport tarfile\nimport time\nimport urllib.error\nimport urllib.request\nimport zipfile\n\ntry:\n    import soundfile as sf\nexcept ImportError:\n    sf = None\n\nREVISION = "70bb2e84b976b7e960aa89f1c648e09c59f894dd"\nFLEURS = f"https://huggingface.co/datasets/google/fleurs/resolve/{REVISION}"\nVCTK = "https://datashare.ed.ac.uk/server/api/core/bitstreams/535f4286-e54c-4038-838c-a02285e32cb2/content"\nVCTK_SIZE = 11_747_302_977\nBASELINE = "1abd3a60c2775b712508dadbae6de1b40f962ee6"\nSALT = "snacka-neural-v2:2026-10-21:"\nMAX_AUDIO_BYTES = 32 * 1024 * 1024\n\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef order(key):\n    return digest((SALT + key).encode())\n\n\ndef split_for(source, group):\n    bucket = int(order(f"{source}:{group}")[:8], 16) % 100\n    if source == "fleurs-sv":\n        return "train" if bucket < 90 else "validation"\n    return "train" if bucket < 80 else "validation" if bucket < 90 else "test"\n\n\ndef request(url, headers=None):\n    for attempt in range(4):\n        try:\n            return urllib.request.urlopen(urllib.request.Request(\n                url, headers={"User-Agent": "SnackaCorpus/2.0", **(headers or {})}\n            ), timeout=90)\n        except (urllib.error.URLError, TimeoutError):\n            if attempt == 3:\n                raise\n            time.sleep(2 ** attempt)\n\n\ndef fetch_small(url, limit=4 * 1024 * 1024):\n    with request(url) as response:\n        data = response.read(limit + 1)\n    if len(data) > limit:\n        raise ValueError("metadata exceeded limit")\n    return data\n\n\nclass RangeReader(io.RawIOBase):\n    """Seek the publisher ZIP without silently downloading its 11.75 GB body."""\n    def __init__(self, url, size):\n        self.url, self.size, self.position = url, size, 0\n        self.cache = OrderedDict()\n        self.chunk_size = 512 * 1024\n        self.transferred = 0\n\n    def seekable(self):\n        return True\n\n    def readable(self):\n        return True\n\n    def tell(self):\n        return self.position\n\n    def seek(self, offset, whence=0):\n        target = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset\n        if target < 0:\n            raise ValueError("negative seek")\n        self.position = target\n        return target\n\n    def block(self, index):\n        if index in self.cache:\n            self.cache.move_to_end(index)\n            return self.cache[index]\n        data = self.fetch_block(index)\n        self.cache[index] = data\n        if len(self.cache) > 16:\n            self.cache.popitem(last=False)\n        return data\n\n    def fetch_block(self, index):\n        start = index * self.chunk_size\n        end = min(start + self.chunk_size, self.size) - 1\n        # Distinct query keys also prevent intermediary caches ignoring Range.\n        url = self.url + f"?snacka_range={start}-{end}"\n        with request(url, {"Range": f"bytes={start}-{end}"}) as response:\n            expected = f"bytes {start}-{end}/{self.size}"\n            if response.status != 206 or response.headers.get("Content-Range") != expected:\n                raise ValueError("server did not honor the exact byte range")\n            data = response.read(end - start + 2)\n        if len(data) != end - start + 1:\n            raise ValueError("truncated/oversized ZIP range")\n        self.transferred += len(data)\n        return data\n\n    def prefetch(self, members):\n        indices = set()\n        for member in members:\n            first = member.header_offset // self.chunk_size\n            last = min(self.size - 1, member.header_offset + member.compress_size + 2048) // self.chunk_size\n            indices.update(range(first, last + 1))\n        ordered = sorted(indices)\n        with ThreadPoolExecutor(max_workers=12) as pool:\n            blocks = list(pool.map(self.fetch_block, ordered))\n        self.cache = OrderedDict(zip(ordered, blocks))\n\n    def read(self, size=-1):\n        end = self.size if size < 0 else min(self.position + size, self.size)\n        chunks = []\n        while self.position < end:\n            index, offset = divmod(self.position, self.chunk_size)\n            block = self.block(index)\n            take = min(end - self.position, len(block) - offset)\n            chunks.append(block[offset:offset + take])\n            self.position += take\n        return b"".join(chunks)\n\n\nclass Corpus:\n    def __init__(self, root):\n        self.root = root\n        self.rows = []\n        self.pcm_seen = {}\n\n    def add(self, data, source, member, group, split, rate, language, **extra):\n        if not data or len(data) > MAX_AUDIO_BYTES:\n            raise ValueError("audio member is empty or exceeds size limit")\n        name = re.sub(r"[^a-zA-Z0-9_.-]", "_", Path(member).stem) + ".flac"\n        relative = Path("audio") / split / source / name\n        target = self.root / relative\n        target.parent.mkdir(parents=True, exist_ok=True)\n        if target.exists():\n            raise FileExistsError(target)\n        # FLAC stores PCM24 at the native rate; source float WAV is quantized.\n        # libsndfile avoids spawning three ffmpeg processes per cloud utterance.\n        if sf is not None:\n            values, actual_rate = sf.read(io.BytesIO(data), dtype="float32", always_2d=True)\n            if actual_rate != rate:\n                raise ValueError(f"unexpected source sample rate: {member}")\n            stream = io.BytesIO()\n            sf.write(stream, values.mean(axis=1), rate, format="FLAC", subtype="PCM_24")\n            converted = stream.getvalue()\n            target.write_bytes(converted)\n            recovered, actual_rate = sf.read(target, dtype="float32")\n            pcm = recovered.astype("<f4").tobytes()\n        else:\n            converted = subprocess.run([\n            "ffmpeg", "-v", "error", "-i", "pipe:0", "-ac", "1",\n            "-c:a", "flac", "-f", "flac", "pipe:1"\n            ], input=data, capture_output=True, check=True).stdout\n            target.write_bytes(converted)\n            probe = json.loads(subprocess.check_output([\n            "ffprobe", "-v", "error", "-show_entries",\n            "stream=sample_rate,channels:format=duration", "-of", "json", str(target)\n            ]))\n            stream = probe["streams"][0]\n            if int(stream["sample_rate"]) != rate or stream["channels"] != 1:\n                raise ValueError(f"unexpected native audio format: {member}")\n            # Decode canonical mono f32 to catch duplicate audio across containers.\n            pcm = subprocess.check_output([\n            "ffmpeg", "-v", "error", "-i", str(target), "-f", "f32le", "pipe:1"\n            ])\n        pcm_sha = digest(pcm)\n        if pcm_sha in self.pcm_seen:\n            previous = self.pcm_seen[pcm_sha]\n            if previous != split:\n                raise ValueError(f"duplicate PCM across splits: {member}")\n            target.unlink()\n            return False\n        self.pcm_seen[pcm_sha] = split\n        self.rows.append(dict(\n            path=relative.as_posix(), source=source, source_member=member,\n            group_id=group, split=split, language=language, native_rate=rate,\n            samples=len(pcm) // 4, seconds=len(pcm) / (4 * rate),\n            source_sha256=digest(data), flac_sha256=digest(converted),\n            pcm_f32_sha256=pcm_sha, license="CC-BY-4.0", **extra\n        ))\n        return True\n\n    def fleurs(self, full=False):\n        metadata = fetch_small(FLEURS + "/data/sv_se/train.tsv")\n        (self.root / "sources" / "fleurs-sv-train.tsv").write_bytes(metadata)\n        mapping = {}\n        for line in metadata.decode().splitlines():\n            fields = line.split("\\t")\n            if len(fields) >= 2:\n                mapping[fields[1]] = fields[0]\n        counts = Counter()\n        caps = {"train": 96, "validation": 12}\n        url = FLEURS + "/data/sv_se/audio/train.tar.gz"\n        with request(url) as response, tarfile.open(fileobj=response, mode="r|gz") as archive:\n            for member in archive:\n                if not member.isfile() or not member.name.endswith(".wav"):\n                    continue\n                sentence = mapping[Path(member.name).name]\n                split = split_for("fleurs-sv", sentence)\n                if not full and counts[split] >= caps[split]:\n                    continue\n                if member.size > MAX_AUDIO_BYTES:\n                    raise ValueError("oversized FLEURS member")\n                with archive.extractfile(member) as audio:\n                    added = self.add(audio.read(), "fleurs-sv", member.name, sentence,\n                                     split, 16000, "sv", revision=REVISION,\n                                     split_claim="sentence-group; not speaker-disjoint")\n                counts[split] += added\n                if sum(counts.values()) % 24 == 0:\n                    print("FLEURS", dict(counts), flush=True)\n                if not full and all(counts[s] >= cap for s, cap in caps.items()):\n                    break\n        if not full and any(counts[s] < cap for s, cap in caps.items()):\n            raise ValueError(f"FLEURS pilot incomplete: {counts}")\n        print("FLEURS complete", dict(counts), flush=True)\n\n    def vctk(self, full=False, training=False, local_archive=None):\n        reader = RangeReader(VCTK, VCTK_SIZE)\n        with zipfile.ZipFile(local_archive or reader) as archive:\n            speakers = {}\n            for member in archive.infolist():\n                match = re.search(r"wav48_silence_trimmed/(p\\d+)/[^/]+_mic2\\.flac$", member.filename)\n                if match and match[1] not in {"p280", "p315"}:\n                    speakers.setdefault(match[1], []).append(member)\n            if not speakers:\n                raise ValueError("VCTK ZIP structure changed")\n            split_map = {speaker: split_for("vctk", speaker) for speaker in speakers}\n            (self.root / "sources" / "vctk-speaker-splits.json").write_text(\n                json.dumps(split_map, sort_keys=True, indent=2) + "\\n")\n            selected = []\n            caps = {"train": 12, "validation": 3, "test": 3}\n            for split, cap in caps.items():\n                candidates = sorted((s for s in speakers if split_map[s] == split), key=order)\n                if not full and len(candidates) < cap:\n                    raise ValueError(f"insufficient VCTK {split} speakers")\n                selected.extend(candidates if full or training else candidates[:cap])\n            for speaker in selected:\n                members = sorted(speakers[speaker], key=lambda m: order(m.filename))\n                chosen = members if full else members[:32 if training else 4]\n                if training and not local_archive:\n                    reader.prefetch(chosen)\n                for member in chosen:\n                    if member.file_size > MAX_AUDIO_BYTES:\n                        raise ValueError("oversized VCTK member")\n                    self.add(archive.read(member), "vctk", member.filename, speaker,\n                             split_map[speaker], 48000, "en", microphone="mic2",\n                             revision="0.92", split_claim="speaker-disjoint")\n                print("VCTK", speaker, split_map[speaker], len(self.rows), flush=True)\n        print(f"VCTK ZIP fetched bytes: {reader.transferred:,} / {VCTK_SIZE:,}", flush=True)\n\n    def finish(self, full, training=False):\n        groups = {}\n        for row in self.rows:\n            key = (row["source"], row["group_id"])\n            if key in groups and groups[key] != row["split"]:\n                raise ValueError("group leakage across splits")\n            groups[key] = row["split"]\n        manifest = "".join(json.dumps(row, sort_keys=True) + "\\n" for row in self.rows)\n        (self.root / "manifest.jsonl").write_text(manifest)\n        summary = dict(schema_version=1, baseline=BASELINE, seed_salt=SALT,\n                       full=full, training=training, clips=len(self.rows), manifest_sha256=digest(manifest.encode()),\n                       strata=[])\n        for source, split in sorted({(r["source"], r["split"]) for r in self.rows}):\n            rows = [r for r in self.rows if (r["source"], r["split"]) == (source, split)]\n            summary["strata"].append(dict(source=source, split=split, clips=len(rows),\n                groups=len({r["group_id"] for r in rows}),\n                seconds=sum(r["seconds"] for r in rows), native_rate=rows[0]["native_rate"]))\n        (self.root / "summary.json").write_text(json.dumps(summary, indent=2) + "\\n")\n        print(json.dumps(summary, indent=2), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--full", action="store_true", help="remove pilot caps; use cloud CPU/disk")\n    parser.add_argument("--training", action="store_true", help="all Swedish train audio and 32 utterances per VCTK speaker")\n    parser.add_argument("--source", choices=["all", "fleurs", "vctk"], default="all")\n    parser.add_argument("--vctk-archive", type=Path, help="optional verified local publisher ZIP on cloud disk")\n    parser.add_argument("--metadata-cache", type=Path, help="publisher metadata staged when a cloud IP is blocked")\n    args = parser.parse_args()\n    if args.output.exists():\n        parser.error("output already exists; select a fresh directory to preserve prior data")\n    args.output.mkdir(parents=True)\n    (args.output / "sources").mkdir()\n    sources = {\n        "fleurs": dict(revision=REVISION, audio=FLEURS + "/data/sv_se/audio/train.tar.gz",\n                       card="https://huggingface.co/datasets/google/fleurs", license="CC-BY-4.0"),\n        "vctk": dict(version="0.92", archive=VCTK, archive_bytes=VCTK_SIZE,\n                     publisher_md5="8a6ba2946b36fcbef0212cad601f4bfa",\n                     card="https://doi.org/10.7488/ds/2645", license="CC-BY-4.0",\n                     archive_validation="per-member ZIP CRC and SHA256; full ZIP MD5 not fetched"),\n    }\n    (args.output / "sources" / "sources.json").write_text(json.dumps(sources, indent=2) + "\\n")\n    for name,url in [\n        ("FLEURS-dataset-card.md",FLEURS + "/README.md"),\n        ("VCTK-README.txt","https://datashare.ed.ac.uk/server/api/core/bitstreams/bb7edd96-5d96-4c0e-8989-1e45597e7b72/content"),\n        ("CC-BY-4.0.txt","https://creativecommons.org/licenses/by/4.0/legalcode.txt")]:\n        cached = args.metadata_cache/name if args.metadata_cache else None\n        (args.output/"sources"/name).write_bytes(cached.read_bytes() if cached and cached.is_file() else fetch_small(url))\n    (args.output / "README.txt").write_text(\n        "Snacka neural v2 raw-audio pilot; NOT extracted PLC training features.\\n"\n        "FLEURS Swedish: Google, CC BY 4.0, pinned revision in sources.json.\\n"\n        "VCTK 0.92: Yamagishi, Veaux, MacDonald, University of Edinburgh, 2019.\\n"\n        "CC BY 4.0, DOI 10.7488/ds/2645. Attribution and license retained in sources/.\\n"\n        "Do not train on audio/test. Validation is for selection; test is for final acceptance.\\n"\n        "FLEURS has sentence-group separation, NOT verified speaker separation.\\n"\n        "Native rates are preserved: Swedish 16 kHz; English VCTK 48 kHz.\\n"\n        "No noise, crops or resampling yet; augment only after splitting.\\n")\n    corpus = Corpus(args.output)\n    if args.vctk_archive:\n        if args.vctk_archive.stat().st_size != VCTK_SIZE:\n            raise ValueError("VCTK archive size differs from pinned publisher metadata")\n        md5 = hashlib.md5()\n        with args.vctk_archive.open("rb") as stream:\n            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n                md5.update(chunk)\n        if md5.hexdigest() != sources["vctk"]["publisher_md5"]:\n            raise ValueError("VCTK publisher MD5 mismatch")\n    if args.source in ("all", "fleurs"):\n        corpus.fleurs(args.full or args.training)\n    if args.source in ("all", "vctk"):\n        corpus.vctk(args.full, args.training, args.vctk_archive)\n    corpus.finish(args.full, args.training)\n\n\nif __name__ == "__main__":\n    main()\n'


In [ ]:
work = Path(tempfile.mkdtemp(prefix="snacka-v2-", dir="/content"))
corpus = work / "corpus"
command = ["python3", str(script), "--output", str(corpus)] + (["--full"] if FULL else [])
subprocess.run(command, check=True)
summary = json.loads((corpus / "summary.json").read_text())
print(json.dumps(summary, indent=2))


In [ ]:
import hashlib, os, time
# A unique run name preserves previously verified exports.
name = "corpus-" + ("full" if FULL else "pilot") + "-" + summary["manifest_sha256"][:12] + "-" + time.strftime("%Y%m%dT%H%M%S") + ".tar.gz"
archive = work / name
with tarfile.open(archive, "w:gz") as tar:
    tar.add(corpus, arcname="corpus")
def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()
expected = sha256_file(archive)
pending = DRIVE_ROOT / (name + ".partial")
shutil.copyfile(archive, pending)
assert sha256_file(pending) == expected, "Drive checksum verification failed"
final = DRIVE_ROOT / name
assert not final.exists()
os.replace(pending, final)
(DRIVE_ROOT / (name + ".sha256")).write_text(expected + "  " + name + "\n")
print("Verified on Drive:", final, expected)
# Delete only this run temporary directory after checksum verification.
shutil.rmtree(work)


Next: implement the native PLC record extractor and QAT trainer described in neural-plc-design.md. Stage feature shards on /content for L4 training, checkpoint to Drive, retrieve weights and verify the exported Rust model. Do not train on VCTK test, and do not change src/opus/. Disconnect/delete the runtime after use.
